# Process a backlog in micro-batches

## Goal

A pipeline that receives new files throughout the day needs to remember what it has already processed. Structured Streaming provides that progress tracking, while the same Rulebricks bulk API evaluates the records in each micro-batch.

This example drains a small file backlog and then stops. The pattern also works for large backlogs because Spark can admit a bounded number of files at a time. It is useful for incremental jobs; a fixed table of independent records can continue using the simpler bulk examples.

| Level | Responsibility |
| --- | --- |
| Spark source and checkpoint | Select new input and remember progress |
| `foreachBatch` | Supply a normal batch DataFrame to your callback |
| Rulebricks chunks | Limit records and bytes sent in each HTTP request |
| Delta output | Keep the source, decisions, and micro-batch ID |

The flow in this example is stateless. Stream checkpoints do not coordinate writes to shared Rulebricks state.

## Setup

Use the [installation guide](../docs/getting-started.md) and the same example flow as [bulk flows](02-bulk-flows.ipynb). It accepts `request_id`, `amount`, and `country` and returns a decision for each record.

Edit the settings below. The local paths are suitable for local Spark; on a cluster, choose storage shared by the driver and executors. Credentials come from `RULEBRICKS_API_KEY` or your secret manager.

`pipeline_id` identifies this checkpoint and output together. Keep it stable when restarting the query. A fresh checkpoint needs a fresh pipeline ID because its micro-batch numbers restart at zero.

In [ ]:
import os

from pyspark.sql import SparkSession, functions

from rulebricks_spark import apply_flow

spark = (
    SparkSession.getActiveSession()
    or SparkSession.builder.appName("streaming-decisions").getOrCreate()
)
api_key = os.environ["RULEBRICKS_API_KEY"]
base_url = "https://rulebricks.acme.com/api/v1"
flow_slug = "order-routing-flow"
flow_version = "1"
output_root = "file:///tmp/rulebricks-examples"
pipeline_id = "order-decisions-demo-v1"
input_path = f"{output_root}/{pipeline_id}/incoming"
checkpoint_path = f"{output_root}/{pipeline_id}/checkpoint"
result_path = f"{output_root}/{pipeline_id}/decisions"

## 1. Create an example backlog

For this walkthrough, Spark writes 18 requests into three input files. Run this setup once for a new pipeline directory. In a real job, a producer supplies files or another streaming source, and you would start at the next step.

In [ ]:
source = spark.range(18).select(
    functions.concat(functions.lit("request-"), functions.col("id").cast("string")).alias(
        "request_id"
    ),
    (functions.col("id") * 10).cast("double").alias("amount"),
    functions.lit("US").alias("country"),
)
source.repartition(3).write.mode("errorifexists").json(input_path)

## 2. Read new files in bounded groups

The source schema is an ordinary Spark schema. `maxFilesPerTrigger=1` admits one input file per micro-batch, so the example processes the backlog in several steps. For other sources, use their admission controls, such as Kafka's `maxOffsetsPerTrigger`.

File counts do not impose a byte limit: choose sensible file sizes as well as the Rulebricks request limits.

In [ ]:
incoming = spark.readStream.schema(source.schema).option("maxFilesPerTrigger", 1).json(input_path)

## 3. Apply the flow to each micro-batch

`foreachBatch` gives this function a normal DataFrame and a monotonically increasing batch ID. We pass only the flow inputs to Rulebricks and retain `source_batch_id` in the output for reconciliation.

| Output column | Meaning |
| --- | --- |
| Source fields | Original request values |
| `source_batch_id` | Spark micro-batch that carried the record |
| `rulebricks_result` | Flow response JSON |
| `rulebricks_meta` | Outcome, errors, and execution details |

Delta's transaction options prevent the same pipeline/batch pair from being appended twice. They do not prevent repeated Rulebricks requests if Spark retries work before the output commits. This is why the example uses an independent, stateless flow.

In [ ]:
def process_batch(batch, batch_id):
    evaluated = apply_flow(
        batch.withColumn("source_batch_id", functions.lit(batch_id)),
        flow_slug,
        version=flow_version,
        input_columns=["request_id", "amount", "country"],
        correlation_id_path="request_id",
        execution_partitions=1,
        max_batch_bytes=5 * 1024 * 1024,
        max_batch_rows=1000,
        api_key=api_key,
        base_url=base_url,
    )
    (
        evaluated.write.format("delta")
        .mode("append")
        .option("txnAppId", pipeline_id)
        .option("txnVersion", batch_id)
        .save(result_path)
    )

## 4. Drain the backlog

`availableNow=True` processes the data available when the query starts, using as many micro-batches as necessary, and then terminates. The checkpoint records progress so a later run can pick up new input.

The source controls how much data enters a micro-batch. The Rulebricks settings inside the callback control how that DataFrame is divided into requests.

In [ ]:
query = (
    incoming.writeStream.foreachBatch(process_batch)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)
query.awaitTermination()

## 5. Inspect the completed run

Read the Delta table to check both processing progress and decision outcomes. A completed stream can still contain failed decisions: those rows stay in the table for investigation rather than disappearing.

In [ ]:
saved = spark.read.format("delta").load(result_path)
assert saved.count() == 18
assert saved.select("request_id").distinct().count() == 18
assert saved.select("source_batch_id").distinct().count() == 3
assert saved.filter("rulebricks_meta.status != 'ok'").count() == 0
saved.groupBy("source_batch_id", "rulebricks_meta.status").count().orderBy("source_batch_id").show()

## 6. Restart from the same checkpoint

Start the query again without recreating input or changing its settings. Since no new files have arrived, this run should leave the result table unchanged.

In [ ]:
restarted = (
    incoming.writeStream.foreachBatch(process_batch)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .start()
)
restarted.awaitTermination()
assert spark.read.format("delta").load(result_path).count() == 18

## Keep processing as data arrives

For a long-running query, replace the available-now trigger with `.trigger(processingTime="30 seconds")` and let your job scheduler manage its lifetime. This remains micro-batch processing; it is not Spark's continuous-processing mode.

Keep the checkpoint, pipeline ID, flow version, and input shape together as job configuration. If you replace the checkpoint, use a new pipeline ID. Coordinate recovery with any downstream effects because the checkpoint and Rulebricks do not share a transaction.

This example appends multiple operations into one analytical table. `resume()` expects one saved operation and cannot recover that combined table directly. If your recovery process needs it, write each micro-batch to its own Delta path instead. See [streaming](../docs/streaming.md) for the tradeoffs and [reliability and audit](../docs/reliability-and-audit.md) for failure handling.